In [20]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

In [21]:
sample_size = 3500
row_df = pd.read_csv("../data/ab_test_dataset.csv")
shuffled_row_df = row_df.sample(frac=1, random_state=42, ignore_index=True)
control_df = shuffled_row_df[shuffled_row_df["group"] == "control"].head(sample_size).reset_index(drop=True)
treatment_df = shuffled_row_df[shuffled_row_df["group"] == "treatment"].head(sample_size).reset_index(drop=True)
df = pd.concat([control_df, treatment_df], axis=0, ignore_index=True)

In [22]:
summary = df.groupby("group")["converted"].agg(["count", "sum", "mean"]).rename(
    columns={"count": "total_users", "sum": "conversions", "mean": "converion_rate"}
)
print(summary)

           total_users  conversions  converion_rate
group                                              
control           3500          796        0.227429
treatment         3500          870        0.248571


In [23]:
count = [summary.loc["treatment", "conversions"], summary.loc["control", "conversions"]]
nobs = [summary.loc["treatment", "total_users"], summary.loc["control", "total_users"]]

z_stat, p_value = proportions_ztest(count, nobs)
print(f"Z-статистика: {z_stat:.3f}")
print(f"p-value: {p_value:.4f}")

Z-статистика: 2.077
p-value: 0.0378


In [ ]:
ci_low, ci_upp = confint_proportions_2indep(
    count1=summary.loc["treatment", "conversions"], nobs1=summary.loc["treatment", "total_users"],
    count2=summary.loc["control", "conversions"], nobs2=summary.loc["control", "total_users"]
)
print(f"95% доверительный интервал разности конверсий: [{ci_low:.4f}, {ci_upp:.4f}]")

95% доверительный интервал разницы конверсий: [0.0012, 0.0411]


# Пояснения к блокноту

Берём sample_size округлённый в большую сторону чем в MDE, перемешиваем выборку на всякий случай, используем z-test так как метрика бинарная, из него получаем p-value, в нашем случае < 0.05, что позволяет отклонить нулевую гипотезу и принять альтернативную, дополнительно получаем доверительный интервал разности конверсий, он поможет оценить размер различий между группами.